Install Required Libraries

In [ ]:
!pip install pdfminer.six

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 80.5 MB/s eta 0:00:00


Upload ZIP Dataset

In [ ]:
from google.colab import files

uploaded = files.upload()  # Upload archive (7).zip

Saving archive (7).zip to archive (7).zip


In [ ]:
import os
os.listdir()

['.config', 'archive (7).zip', 'sample_data']

Extract Dataset

In [ ]:
import zipfile

zip_path = "archive (7).zip"   # Change if filename differs
extract_path = "resume_dataset"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset Extracted Successfully!")


Dataset Extracted Successfully!


Import Libraries

In [ ]:
import os
import pandas as pd
from pdfminer.high_level import extract_text
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


Extract Resume Text from PDFs

In [ ]:
resume_folder = os.path.join(
    extract_path,
    "HireAMLE",
    "dataset",
    "trainResumes"
)

resume_texts = []
resume_names = []

for file in os.listdir(resume_folder):
    if file.endswith(".pdf"):
        file_path = os.path.join(resume_folder, file)
        try:
            text = extract_text(file_path)
            if text and len(text.strip()) > 0:
                resume_texts.append(text)
                resume_names.append(file)
        except:
            continue

print("Total Resumes Processed:", len(resume_texts))


Total Resumes Processed: 90


Extract Job Description

In [ ]:
job_description_path = os.path.join(
    extract_path,
    "HireAMLE",
    "dataset",
    "Job description.pdf"
)

job_description = extract_text(job_description_path)

print("Job Description Loaded Successfully!")


Job Description Loaded Successfully!


Apply TF-IDF + Cosine Similarity

In [ ]:
documents = resume_texts + [job_description]

vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)

tfidf_matrix = vectorizer.fit_transform(documents)

similarity_scores = cosine_similarity(
    tfidf_matrix[-1],
    tfidf_matrix[:-1]
)

scores = similarity_scores[0]

# Convert to Percentage
percentage_scores = scores * 100


Rank Candidates

In [ ]:
ranking_df = pd.DataFrame({
    "Candidate_Name": resume_names,
    "Match_Score_%": percentage_scores
})

ranking_df = ranking_df.sort_values(
    by="Match_Score_%",
    ascending=False
).reset_index(drop=True)

ranking_df.head(15)


,Candidate_Name,Match_Score_%
0,candidate_134.pdf,22.404539
1,candidate_137.pdf,20.327925
2,candidate_063.pdf,20.058371
3,candidate_121.pdf,19.680439
4,candidate_096.pdf,17.653033
5,candidate_093.pdf,17.494831
6,candidate_050.pdf,17.161868
7,candidate_133.pdf,16.903473
8,candidate_034.pdf,16.810700
9,candidate_009.pdf,16.580583


Save & Download Results

In [ ]:
output_file = "candidate_ranking_results.csv"
ranking_df.to_csv(output_file, index=False)

from google.colab import files
files.download(output_file)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>